In [ ]:


# EDA PROJECT: EXPLORATORY ANALYSIS OF HOUSE PRICES


# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from google.colab import files

sns.set_style("whitegrid")

# 2. LOAD DATASET
uploaded = files.upload()
df = pd.read_csv("train.csv")

print("DATASET LOADED SUCCESSFULLY!")
print("Shape:", df.shape)
print(df.head())
print(df.info())

# 3. CHECK MISSING VALUES
missing = df.isnull().sum()
print("\nMissing Values:")
print(missing[missing > 0].sort_values(ascending=False))

# 4. CLASSIFY VARIABLES
numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(include="object").columns

print("\nNumerical Columns:", len(numeric_cols))
print("Categorical Columns:", len(categorical_cols))
print("OverallQual is an ordinal variable.")

# 5. DESCRIPTIVE STATISTICS
print("\nStatistical Summary:")
print(df.describe())

for col in ["SalePrice", "GrLivArea", "LotArea"]:
    print("\nStatistics of", col)
    print("Mean:", df[col].mean())
    print("Median:", df[col].median())
    print("Mode:", df[col].mode()[0])
    print("Standard Deviation:", df[col].std())
    print("Variance:", df[col].var())
    print("Range:", df[col].max() - df[col].min())
    print("IQR:", df[col].quantile(.75) - df[col].quantile(.25))
    print("Skewness:", df[col].skew())
    print("Kurtosis:", df[col].kurt())

# 6. DISTRIBUTION OF SALE PRICE
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(df["SalePrice"], kde=True, ax=ax[0])
ax[0].set_title("Sale Price Distribution")

sns.boxplot(x=df["SalePrice"], ax=ax[1])
ax[1].set_title("Sale Price Boxplot")

sns.histplot(np.log1p(df["SalePrice"]), kde=True, ax=ax[2])
ax[2].set_title("Log Sale Price")

plt.tight_layout()
plt.show()

# 7. PRICE BY OVERALL QUALITY
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="OverallQual", y="SalePrice")
plt.title("Sale Price by Overall Quality")
plt.show()

# 8. PRICE BY NEIGHBORHOOD
order = df.groupby("Neighborhood")["SalePrice"].median().sort_values().index

plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x="Neighborhood", y="SalePrice", order=order)
plt.xticks(rotation=90)
plt.title("Sale Price by Neighborhood")
plt.show()

# 9. COUNT PLOTS
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x="OverallQual", ax=ax[0])
ax[0].set_title("Overall Quality Count")

sns.countplot(data=df, x="HouseStyle", ax=ax[1])
ax[1].tick_params(axis="x", rotation=45)
ax[1].set_title("House Style Count")

plt.tight_layout()
plt.show()

# 10. CORRELATION AND COVARIANCE
corr = df.select_dtypes(include=np.number).corr()

print("\nTop 10 Features Related to SalePrice:")
print(corr["SalePrice"].drop("SalePrice").abs().sort_values(ascending=False).head(10))

print("\nCovariance:")
print(df[["SalePrice", "OverallQual", "GrLivArea", "GarageCars"]].cov())

# 11. SCATTER PLOTS
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

sns.scatterplot(data=df, x="GrLivArea", y="SalePrice", ax=ax[0])
ax[0].set_title("Living Area vs Sale Price")

sns.scatterplot(data=df, x="TotalBsmtSF", y="SalePrice", ax=ax[1])
ax[1].set_title("Basement Area vs Sale Price")

sns.scatterplot(data=df, x="YearBuilt", y="SalePrice", ax=ax[2])
ax[2].set_title("Year Built vs Sale Price")

plt.tight_layout()
plt.show()

# 12. CORRELATION HEATMAP
top10 = corr["SalePrice"].abs().sort_values(ascending=False).head(10).index

plt.figure(figsize=(10, 8))
sns.heatmap(df[top10].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Top 10 Features Correlation Heatmap")
plt.show()

# 13. OUTLIER DETECTION USING IQR AND Z-SCORE
for col in ["GrLivArea", "SalePrice"]:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    iqr_outliers = df[
        (df[col] < Q1 - 1.5 * IQR) |
        (df[col] > Q3 + 1.5 * IQR)
    ]

    z_scores = np.abs(stats.zscore(df[col]))
    z_outliers = df[z_scores > 3]

    print("\nOutliers in", col)
    print("IQR Outliers:", len(iqr_outliers))
    print("Z-Score Outliers:", len(z_outliers))

# 14. DATA QUALITY CHECKS
print("\nDuplicate Rows:", df.duplicated().sum())

print("YearBuilt greater than YrSold:",
      (df["YearBuilt"] > df["YrSold"]).sum())

# Clean missing values in a separate copy
df_clean = df.copy()

none_cols = [
    "Alley", "PoolQC", "Fence", "FireplaceQu",
    "GarageType", "BsmtQual", "BsmtCond",
    "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "GarageFinish", "GarageQual", "GarageCond",
    "MiscFeature"
]

for col in none_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna("None")

for col in df_clean.select_dtypes(include=np.number).columns:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in df_clean.select_dtypes(include="object").columns:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print("Remaining Missing Values:", df_clean.isnull().sum().sum())

# 15. FINAL DASHBOARD
fig, ax = plt.subplots(2, 2, figsize=(15, 10))

# Line Plot
year_price = df.groupby("YearBuilt")["SalePrice"].mean()
ax[0, 0].plot(year_price.index, year_price.values)
ax[0, 0].set_title("Average Price by Year Built")
ax[0, 0].set_xlabel("Year Built")
ax[0, 0].set_ylabel("Average Sale Price")

# Bar Plot
top_neigh = df.groupby("Neighborhood")["SalePrice"].mean().nlargest(10)
top_neigh.sort_values().plot(kind="barh", ax=ax[0, 1])
ax[0, 1].set_title("Top 10 Neighborhoods")

# Histogram
sns.histplot(df["SalePrice"], kde=True, ax=ax[1, 0])
ax[1, 0].axvline(df["SalePrice"].mean(), color="red", label="Mean")
ax[1, 0].axvline(df["SalePrice"].median(), color="green", label="Median")
ax[1, 0].legend()
ax[1, 0].set_title("Sale Price Distribution")

# Scatter Plot
sns.scatterplot(
    data=df, x="GrLivArea", y="SalePrice",
    hue="OverallQual", ax=ax[1, 1]
)
ax[1, 1].set_title("Area vs Price by Quality")

plt.suptitle("HOUSE PRICE EDA SUMMARY")
plt.tight_layout()
plt.savefig("eda_dashboard.png")
plt.show()

print("\nEDA PROJECT COMPLETED SUCCESSFULLY!")

print("""
CONCLUSION:
1. House prices vary according to house quality.
2. Living area is related to sale price.
3. Neighborhood affects house prices.
4. Correlation helps identify relationships.
5. Outliers and missing values need proper analysis.
""")

# DOWNLOAD FINAL DASHBOARD
files.download("eda_dashboard.png")


In [ ]:

# ADDITIONAL ANALYSIS - PDF REQUIREMENTS


# 1. VARIABLE CLASSIFICATION TABLE

ordinal_cols = ["OverallQual", "ExterQual", "KitchenQual"]

variable_summary = []

for col in df.columns:
    if col in ordinal_cols:
        typ = "Ordinal"
    elif pd.api.types.is_numeric_dtype(df[col]):
        typ = "Numerical"
    else:
        typ = "Categorical"

    variable_summary.append({
        "Column": col,
        "Type": typ,
        "Unique Values": df[col].nunique(),
        "Examples": str(df[col].dropna().unique()[:3])
    })

variable_table = pd.DataFrame(variable_summary)
display(variable_table.head(20))

# 2. NEIGHBORHOOD AND HOUSE STYLE FREQUENCY

print("Neighborhood Frequency:")
print(df["Neighborhood"].value_counts())

print("\nHouse Style Frequency:")
print(df["HouseStyle"].value_counts())

print("\nMost Common Neighborhood:")
print(df["Neighborhood"].mode()[0])

print("\nMost Common House Style:")
print(df["HouseStyle"].mode()[0])

# 3. TOP 3 CORRELATION AND COVARIANCE

top3_features = (
    corr["SalePrice"]
    .drop("SalePrice")
    .abs()
    .sort_values(ascending=False)
    .head(3).index
)

print("\nTop 3 Correlated Features:")
for col in top3_features:
    print(
        col,
        "Correlation:", corr.loc[col, "SalePrice"],
        "Covariance:", df[col].cov(df["SalePrice"])
    )

# 4. SPEARMAN CORRELATION

print("\nSpearman Correlation:")
print(df[["OverallQual", "SalePrice"]].corr(method="spearman"))

# 5. SCATTER PLOT WITH UNUSUAL HOUSES

unusual = (df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)

plt.figure(figsize=(9, 5))

sns.scatterplot(
    data=df,
    x="GrLivArea",
    y="SalePrice",
    color="blue",
    label="Normal observations"
)

plt.scatter(
    df.loc[unusual, "GrLivArea"],
    df.loc[unusual, "SalePrice"],
    color="red",
    label="Unusual houses"
)

plt.title("Unusual Houses: Living Area vs Sale Price")
plt.xlabel("Above Ground Living Area")
plt.ylabel("Sale Price")
plt.legend()
plt.show()

print("Number of unusual houses:", unusual.sum())

# 6. PAIRPLOT

selected = [
    "SalePrice",
    "GrLivArea",
    "TotalBsmtSF",
    "GarageArea",
    "YearBuilt"
]

sns.pairplot(
    df[selected],
    diag_kind="kde",
    corner=True
)

plt.show()

# 7. IMPROVED CORRELATION HEATMAP

top10 = (
    corr["SalePrice"]
    .abs()
    .sort_values(ascending=False)
    .head(10).index
)

corr_top = df[top10].corr()

mask = np.triu(
    np.ones_like(corr_top, dtype=bool)
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    corr_top,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    mask=mask
)

plt.title("Top 10 Features Correlated with SalePrice")
plt.show()

# 8. IQR AND Z-SCORE OUTLIER OVERLAP

for col in ["GrLivArea", "SalePrice"]:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    iqr_indices = set(
        df.index[(df[col] < lower) | (df[col] > upper)]
    )

    z = np.abs(stats.zscore(df[col]))
    z_indices = set(df.index[z > 3])

    overlap = iqr_indices.intersection(z_indices)

    print("\nOutlier Analysis:", col)
    print("IQR Outliers:", len(iqr_indices))
    print("Z-Score Outliers:", len(z_indices))
    print("Common Outliers:", len(overlap))

# 9. DATA QUALITY CHECKS

print("\nDuplicate Rows:", df.duplicated().sum())

# Check extra spaces and different letter casing
for col in categorical_cols:

    values = df[col].dropna().astype(str)

    extra_spaces = (values != values.str.strip()).sum()

    case_variations = (
        values.groupby(values.str.lower()).nunique() > 1
    ).sum()

    if extra_spaces > 0 or case_variations > 0:
        print(
            col,
            "Extra spaces:", extra_spaces,
            "Case variations:", case_variations
        )

# Check impossible values
area_cols = [
    "GrLivArea",
    "LotArea",
    "TotalBsmtSF",
    "GarageArea"
]

for col in area_cols:
    print(
        col,
        "Negative values:",
        (df[col] < 0).sum()
    )

print(
    "YearBuilt greater than YrSold:",
    (df["YearBuilt"] > df["YrSold"]).sum()
)

# 10. HANDLE LOTFRONTAGE MISSING VALUES
quality_df = df.copy()

quality_df["LotFrontage"] = (
    quality_df.groupby("Neighborhood")["LotFrontage"]
    .transform(lambda x: x.fillna(x.median()))
)

quality_df["LotFrontage"] = (
    quality_df["LotFrontage"].fillna(
        quality_df["LotFrontage"].median()
    )
)

print(
    "\nLotFrontage Missing After Cleaning:",
    quality_df["LotFrontage"].isnull().sum()
)

print("\nADDITIONAL EDA ANALYSIS COMPLETED!")
